# 02 — Data Quality & Cleaning (PySpark)

## Objective
Clean the five insurance tables, validate label integrity,
and enforce insurance business rules before any feature work.

## What Is Different About Insurance Cleaning

1. **The label itself can be corrupt.**
   In CLV and Loan Default the label was derived from clean sources.
   Here we audit claim_amount against IDV, policy dates and the
   claims table before trusting it.

2. **Domain rules, not just statistics.**
   - Own-damage/theft/fire/calamity payout can never exceed IDV
   - A claim must fall inside its policy period
   - NCB must be one of 0/20/25/35/45/50 and is 0 after any claim

3. **Reconstruct instead of impute.**
   total_premium is an arithmetic function of its components,
   so a missing value can be recomputed exactly.

4. **Do NOT cap the tail.**
   In CLV we capped transactions at P99.9. Here the large claims
   ARE the business. Only the IDV rule (a real contractual limit)
   is applied to claim_amount.

## Leakage Map (decided now, enforced in Notebook 04)
Post-claim columns — exist only AFTER an accident, never features:
  claim_type, claim_date, claim_month, claim_year, surveyor_assessment,
  settlement_days, is_fraud_suspected, monsoon_claim

Generator internals — never available in real life:
  claim_labels.claim_prob  (the TRUE claim probability)
  -> saved separately as an "oracle" file for evaluation only

This is the third project where leakage is the central risk.
Deciding the map before feature engineering is the lesson.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_PATH   = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
DB_PATH    = os.path.abspath("../data/insurance.db")
DB_URL     = f"jdbc:sqlite:{DB_PATH}"
PROCESSED  = "../data/processed"
LOCAL_TEMP = os.path.abspath("../data/spark_temp")

os.makedirs(PROCESSED,  exist_ok=True)
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Insurance_Cleaning")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def read_table(name):
    return (
        spark.read.format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

policyholders = read_table("policyholders")
vehicles      = read_table("vehicles")
policies      = read_table("policies")
claims        = read_table("claims")
labels        = read_table("claim_labels").withColumn(
    "has_claim", F.col("has_claim").cast(IntegerType())
)

baseline = labels.agg(
    F.avg("has_claim").alias("claim_rate"),
    F.sum("claim_amount").alias("total_claims"),
    F.avg("claim_amount").alias("mean_claim"),
).collect()[0]

INITIAL_CLAIM_RATE   = baseline["claim_rate"] * 100
INITIAL_TOTAL_CLAIMS = baseline["total_claims"]

print("All tables loaded ✅")
print(f"  policyholders : {policyholders.count():>8,}")
print(f"  vehicles      : {vehicles.count():>8,}")
print(f"  policies      : {policies.count():>8,}")
print(f"  claims        : {claims.count():>8,}")
print(f"  claim_labels  : {labels.count():>8,}")
print(f"\nBaseline (must stay stable through cleaning):")
print(f"  Claim rate    : {INITIAL_CLAIM_RATE:.4f}%")
print(f"  Total claims  : ₹{INITIAL_TOTAL_CLAIMS:,.0f}")
print(f"  Mean claim    : ₹{baseline['mean_claim']:,.2f} (all policies)")

All tables loaded ✅
  policyholders :  100,500
  vehicles      :  100,000
  policies      :  100,000
  claims        :   36,478
  claim_labels  :  100,000

Baseline (must stay stable through cleaning):
  Claim rate    : 36.4780%
  Total claims  : ₹8,339,565,569
  Mean claim    : ₹83,395.66 (all policies)


In [2]:
print("NULL AUDIT — Classify Every Null Before Touching Anything\n")

def null_audit(df, table_name):
    total  = df.count()
    counts = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    print(f"── {table_name.upper()} ({total:,} rows) ──")
    found = False
    for col, n in counts.items():
        if n > 0:
            found = True
            print(f"  {col:<28} {n:>8,}  ({n/total*100:.2f}%)")
    if not found:
        print("  No nulls found ✅")
    print()

null_audit(policyholders, "policyholders")
null_audit(vehicles,      "vehicles")
null_audit(policies,      "policies")
null_audit(claims,        "claims")
null_audit(labels,        "claim_labels")

print("""NULL CLASSIFICATION

TYPE 1 — Quality nulls (genuine missing data, impute):
  gender, ncb_percentage, vehicle_age_years, engine_cc

TYPE 2 — PII (never needed for modeling, drop from cleaned output):
  phone, email

TYPE 3 — CASCADED nulls (new in this project):
  A null in a parent column makes every derived column null.
  Expect idv, base_od_premium, ncb_discount and addon_premium
  to show roughly the SAME count in policies, because
  base_od_premium = idv x rate, and the others derive from it.
  total_premium adds its own ~1.5% random nulls on top.
  Fixing the parent (or removing the rows) fixes the cascade.
""")

# ── Diagnostic 1: is the label corrupt where IDV is missing? ──
print("=" * 70)
print("DIAGNOSTIC 1 — Claim amounts where IDV is missing\n")

idv_null_ids = policies.filter(F.col("idv").isNull()).select("policy_id").distinct()
n_idv_null   = idv_null_ids.count()

lab_idv_null = labels.join(idv_null_ids, on="policy_id", how="inner")
lab_idv_ok   = labels.join(idv_null_ids, on="policy_id", how="left_anti")

def claim_profile(df, label):
    n    = df.count()
    rate = df.agg(F.avg("has_claim")).collect()[0][0] * 100
    s    = df.filter(F.col("claim_amount") > 0).agg(
        F.count("*").alias("n"),
        F.percentile_approx("claim_amount", 0.5, 1000).alias("median"),
        F.min("claim_amount").alias("min"),
        F.max("claim_amount").alias("max"),
        F.avg((F.col("claim_amount") < 2000).cast("int")).alias("pct_lt_2k"),
    ).collect()[0]
    print(f"  {label:<26} policies={n:>7,} claim_rate={rate:5.2f}% "
          f"claims={s['n']:>6,}")
    print(f"  {'':<26} median=₹{(s['median'] or 0):>10,.0f} "
          f"min=₹{(s['min'] or 0):>9,.0f} max=₹{(s['max'] or 0):>12,.0f} "
          f"below ₹2K: {(s['pct_lt_2k'] or 0)*100:5.1f}%")
    return s

s_null = claim_profile(lab_idv_null, "IDV missing")
s_ok   = claim_profile(lab_idv_ok,   "IDV present")

IDV_ARTIFACT = bool(
    s_null["n"] and s_null["n"] > 0
    and s_null["median"] is not None
    and s_null["median"] < 2000
)
print(f"\n  Policies with null IDV : {n_idv_null:,}")
print(f"  Label artifact confirmed: {IDV_ARTIFACT}")
if IDV_ARTIFACT:
    print("  -> Claims on null-IDV policies sit near a ₹500 floor.")
    print("     The LABEL is corrupt for these rows, so imputing IDV")
    print("     would not repair it. Cell 6 excludes these policies.")
else:
    print("  -> Artifact NOT confirmed. Cell 6 will impute IDV instead.")

# ── Diagnostic 2: is missing vehicle age informative? ─────────
print("\n" + "=" * 70)
print("DIAGNOSTIC 2 — Is a missing vehicle_age_years informative?\n")

age_null_ph = vehicles.filter(
    F.col("vehicle_age_years").isNull()
).select("policyholder_id").distinct()

lab_pol = labels.join(
    policies.select("policy_id", "policyholder_id"), on="policy_id"
)
r_null = lab_pol.join(age_null_ph, on="policyholder_id", how="inner") \
                .agg(F.avg("has_claim")).collect()[0][0] * 100
r_ok   = lab_pol.join(age_null_ph, on="policyholder_id", how="left_anti") \
                .agg(F.avg("has_claim")).collect()[0][0] * 100

print(f"  Claim rate, vehicle age MISSING : {r_null:.2f}%")
print(f"  Claim rate, vehicle age PRESENT : {r_ok:.2f}%")
print(f"  Difference                      : {r_null - r_ok:+.2f} pp")
print("  If the gap is large, missingness carries signal -> we keep a")
print("  vehicle_age_missing flag (same idea as has_credit_history).")

NULL AUDIT — Classify Every Null Before Touching Anything

── POLICYHOLDERS (100,500 rows) ──
  gender                          2,094  (2.08%)
  ncb_percentage                  2,016  (2.01%)
  phone                           2,016  (2.01%)
  email                           4,975  (4.95%)

── VEHICLES (100,000 rows) ──
  vehicle_age_years               2,014  (2.01%)
  engine_cc                       2,990  (2.99%)
  idv                             2,022  (2.02%)

── POLICIES (100,000 rows) ──
  idv                             2,022  (2.02%)
  base_od_premium                 2,022  (2.02%)
  addon_premium                   2,022  (2.02%)
  ncb_discount                    2,022  (2.02%)
  total_premium                   3,450  (3.45%)

── CLAIMS (36,478 rows) ──
  No nulls found ✅

── CLAIM_LABELS (100,000 rows) ──
  No nulls found ✅

NULL CLASSIFICATION

TYPE 1 — Quality nulls (genuine missing data, impute):
  gender, ncb_percentage, vehicle_age_years, engine_cc

TYPE 2 — PII (never ne

In [3]:
print("STEP 1 — REMOVE DUPLICATES\n")

before = {
    "policyholders": policyholders.count(),
    "vehicles"     : vehicles.count(),
    "policies"     : policies.count(),
    "claims"       : claims.count(),
    "claim_labels" : labels.count(),
}

policyholders = policyholders.dropDuplicates().dropDuplicates(["policyholder_id"])
vehicles      = vehicles.dropDuplicates().dropDuplicates(["vehicle_id"])
policies      = policies.dropDuplicates().dropDuplicates(["policy_id"])
claims        = claims.dropDuplicates().dropDuplicates(["claim_id"])
labels        = labels.dropDuplicates().dropDuplicates(["policy_id"])

after = {
    "policyholders": policyholders.count(),
    "vehicles"     : vehicles.count(),
    "policies"     : policies.count(),
    "claims"       : claims.count(),
    "claim_labels" : labels.count(),
}

print(f"{'Table':<16} {'Before':>10} {'After':>10} {'Removed':>10}")
print("-" * 50)
for t in before:
    print(f"  {t:<14} {before[t]:>10,} {after[t]:>10,} "
          f"{before[t]-after[t]:>10,}")

multi_claim = claims.groupBy("policy_id").count().filter(F.col("count") > 1).count()
print(f"\nPolicies with more than one claim row : {multi_claim:,}")
print("(must be 0 — the target is one amount per policy)")

rate_now = labels.agg(F.avg("has_claim")).collect()[0][0] * 100
print(f"\nClaim rate: {INITIAL_CLAIM_RATE:.4f}% -> {rate_now:.4f}%")

STEP 1 — REMOVE DUPLICATES

Table                Before      After    Removed
--------------------------------------------------
  policyholders     100,500    100,000        500
  vehicles          100,000    100,000          0
  policies          100,000    100,000          0
  claims             36,478     36,478          0
  claim_labels      100,000    100,000          0

Policies with more than one claim row : 0
(must be 0 — the target is one amount per policy)

Claim rate: 36.4780% -> 36.4780%


In [4]:
print("STEP 2 — FIX TYPES, DATES AND EXPOSURE\n")

OBS_DATE = "2024-12-31"

# ── Dates (SQLite hands them to Spark as strings) ─────────────
policies = (
    policies
    .withColumn("policy_start_date",
                F.to_date(F.col("policy_start_date"), "yyyy-MM-dd"))
    .withColumn("policy_end_date",
                F.to_date(F.col("policy_end_date"),   "yyyy-MM-dd"))
)
claims = claims.withColumn(
    "claim_date", F.to_date(F.col("claim_date"), "yyyy-MM-dd")
)

# ── Derived policy calendar features + exposure ───────────────
policies = (
    policies
    .withColumn("policy_start_year",    F.year("policy_start_date"))
    .withColumn("policy_start_month",   F.month("policy_start_date"))
    .withColumn("policy_start_quarter", F.quarter("policy_start_date"))
    .withColumn("policy_term_days",
                F.datediff("policy_end_date", "policy_start_date"))
    .withColumn(
        "exposure_years",
        F.round(F.least(
            F.lit(1.0),
            F.datediff(
                F.least(F.col("policy_end_date"), F.to_date(F.lit(OBS_DATE))),
                F.col("policy_start_date")
            ) / F.lit(365.0)
        ), 4)
    )
)

date_nulls = {
    "policy_start_date": policies.filter(F.col("policy_start_date").isNull()).count(),
    "policy_end_date"  : policies.filter(F.col("policy_end_date").isNull()).count(),
    "claim_date"       : claims.filter(F.col("claim_date").isNull()).count(),
}
print(f"Nulls after date casting (must be 0): {date_nulls}")

print("\nExposure check:")
policies.select(
    F.min("exposure_years").alias("min_exposure"),
    F.max("exposure_years").alias("max_exposure"),
    F.round(F.avg((F.col("exposure_years") < 1.0).cast("int")) * 100, 2)
        .alias("pct_partial_year"),
).show()
print("Policy term lengths (days):")
policies.groupBy("policy_term_days").count().orderBy("policy_term_days").show()

# ── 0/1 flags: keep as integers, validate the domain ──────────
flag_policies = ["has_zero_depreciation", "has_engine_protection",
                 "has_roadside_assistance", "has_consumables"]
for c in flag_policies:
    policies = policies.withColumn(c, F.col(c).cast(IntegerType()))
vehicles = vehicles.withColumn("is_modified", F.col("is_modified").cast(IntegerType()))
claims   = (claims
            .withColumn("is_fraud_suspected", F.col("is_fraud_suspected").cast(IntegerType()))
            .withColumn("monsoon_claim",      F.col("monsoon_claim").cast(IntegerType())))

def count_non_binary(df, cols):
    return {c: df.filter(F.col(c).isNotNull() & ~F.col(c).isin(0, 1)).count()
            for c in cols}

print("Non-binary values in flag columns (must all be 0):")
print("  policies :", count_non_binary(policies, flag_policies))
print("  vehicles :", count_non_binary(vehicles, ["is_modified"]))
print("  claims   :", count_non_binary(claims, ["is_fraud_suspected", "monsoon_claim"]))
print("  labels   :", count_non_binary(labels, ["has_claim"]))

bad_monsoon = claims.filter(
    (F.col("monsoon_claim") == 1) != F.col("claim_month").isin(7, 8, 9)
).count()
print(f"\nmonsoon_claim inconsistent with claim_month: {bad_monsoon:,} (must be 0)")

STEP 2 — FIX TYPES, DATES AND EXPOSURE

Nulls after date casting (must be 0): {'policy_start_date': 0, 'policy_end_date': 0, 'claim_date': 0}

Exposure check:
+------------+------------+----------------+
|min_exposure|max_exposure|pct_partial_year|
+------------+------------+----------------+
|         1.0|         1.0|             0.0|
+------------+------------+----------------+

Policy term lengths (days):
+----------------+-----+
|policy_term_days|count|
+----------------+-----+
|             365|58020|
|             366|41980|
+----------------+-----+

Non-binary values in flag columns (must all be 0):
  policies : {'has_zero_depreciation': 0, 'has_engine_protection': 0, 'has_roadside_assistance': 0, 'has_consumables': 0}
  vehicles : {'is_modified': 0}
  claims   : {'is_fraud_suspected': 0, 'monsoon_claim': 0}
  labels   : {'has_claim': 0}

monsoon_claim inconsistent with claim_month: 0 (must be 0)


In [5]:
print("STEP 3 — IMPOSSIBLE VALUES, EXCLUSIONS AND IMPUTATION\n")

# ══════════════════════════════════════════════════════════════
# A. Exclude policies whose LABEL is corrupt (null IDV artifact)
# ══════════════════════════════════════════════════════════════
print("── A. Corrupt-label exclusion ──")
if IDV_ARTIFACT:
    drop_policies = policies.filter(F.col("idv").isNull()) \
                            .select("policy_id", "policyholder_id")
    drop_pol_ids  = drop_policies.select("policy_id")
    drop_ph_ids   = drop_policies.select("policyholder_id")
    n_drop        = drop_policies.count()

    policies      = policies.join(drop_pol_ids, on="policy_id",        how="left_anti")
    vehicles      = vehicles.join(drop_ph_ids,  on="policyholder_id",  how="left_anti")
    policyholders = policyholders.join(drop_ph_ids, on="policyholder_id", how="left_anti")
    claims        = claims.join(drop_pol_ids,   on="policy_id",        how="left_anti")
    labels        = labels.join(drop_pol_ids,   on="policy_id",        how="left_anti")

    print(f"  Excluded {n_drop:,} policies (all five tables kept consistent)")
    print("  Missingness in IDV is random, so dropping ALL of them")
    print("  (claimants and non-claimants) does not bias the claim rate.")
else:
    print("  Artifact not confirmed — no exclusion.")

# ══════════════════════════════════════════════════════════════
# B. policyholders
# ══════════════════════════════════════════════════════════════
print("\n── B. policyholders ──")

bad_age = policyholders.filter(
    (F.col("age") < 18) | (F.col("age") > 85)
).count()
policyholders = policyholders.withColumn(
    "age",
    F.when((F.col("age") < 18) | (F.col("age") > 85), None)
     .otherwise(F.col("age"))
)
age_med = (policyholders.filter(F.col("age").isNotNull())
           .groupBy("occupation")
           .agg(F.percentile_approx("age", 0.5, 1000).alias("age_med")))
policyholders = (policyholders.join(age_med, on="occupation", how="left")
                 .withColumn("age", F.coalesce(F.col("age"), F.col("age_med")))
                 .drop("age_med"))
print(f"  Impossible ages (<18 or >85) -> null -> occupation median: {bad_age:,}")

policyholders = policyholders.fillna({"gender": "Not Specified"})

# NCB: validate slabs, flag missing, impute with a business rule
valid_ncb   = [0, 20, 25, 35, 45, 50]
invalid_ncb = policyholders.filter(
    F.col("ncb_percentage").isNotNull() & ~F.col("ncb_percentage").isin(valid_ncb)
).count()
ncb_conflict = policyholders.filter(
    (F.col("prior_claims_count") > 0) & (F.col("ncb_percentage") > 0)
).count()
print(f"  NCB outside valid slabs                : {invalid_ncb:,}")
print(f"  NCB > 0 despite prior claims (rule: 0) : {ncb_conflict:,}")

policyholders = policyholders.withColumn(
    "ncb_missing", F.col("ncb_percentage").isNull().cast("int")
)
ncb_median_clean = (policyholders
    .filter((F.col("prior_claims_count") == 0) & F.col("ncb_percentage").isNotNull())
    .agg(F.percentile_approx("ncb_percentage", 0.5, 1000)).collect()[0][0])
policyholders = policyholders.withColumn(
    "ncb_percentage",
    F.when(F.col("ncb_percentage").isNotNull(), F.col("ncb_percentage"))
     .when(F.col("prior_claims_count") > 0, F.lit(0.0))
     .otherwise(F.lit(float(ncb_median_clean)))
)
print(f"  NCB imputed: 0 if prior claims, else clean-history median "
      f"({ncb_median_clean:.0f}%)")

# Income: cap at P1/P99 (genuine outliers, unlike claim amounts)
inc_q = policyholders.approxQuantile("monthly_income", [0.01, 0.99], 0.01)
policyholders = policyholders.withColumn(
    "monthly_income",
    F.when(F.col("monthly_income") < inc_q[0], inc_q[0])
     .when(F.col("monthly_income") > inc_q[1], inc_q[1])
     .otherwise(F.col("monthly_income"))
)
print(f"  Income capped at P1 ₹{inc_q[0]:,.0f} / P99 ₹{inc_q[1]:,.0f}")

policyholders = policyholders.drop("phone", "email")
print("  Dropped PII columns: phone, email")

# ══════════════════════════════════════════════════════════════
# C. vehicles
# ══════════════════════════════════════════════════════════════
print("\n── C. vehicles ──")

# Flag BEFORE imputing so the information survives
vehicles = vehicles.withColumn(
    "vehicle_age_missing", F.col("vehicle_age_years").isNull().cast("int")
)
age_med_v = (vehicles.filter(F.col("vehicle_age_years").isNotNull())
             .groupBy("vehicle_segment")
             .agg(F.percentile_approx("vehicle_age_years", 0.5, 1000).alias("age_med")))
vehicles = (vehicles.join(age_med_v, on="vehicle_segment", how="left")
            .withColumn("vehicle_age_years",
                        F.coalesce(F.col("vehicle_age_years"), F.col("age_med")))
            .drop("age_med"))

# registration_year was written as 2024 for rows with unknown age,
# which contradicts any imputed age. Recompute it from the cleaned age.
vehicles = vehicles.withColumn(
    "registration_year",
    (F.lit(2024) - F.col("vehicle_age_years")).cast(IntegerType())
)
print("  vehicle_age_years imputed (segment median) + vehicle_age_missing flag")
print("  registration_year recomputed from cleaned age")

cc_group = (vehicles.filter(F.col("engine_cc").isNotNull())
            .groupBy("vehicle_type", "vehicle_segment")
            .agg(F.percentile_approx("engine_cc", 0.5, 1000).alias("cc_med")))
cc_global = (vehicles.filter(F.col("engine_cc").isNotNull())
             .agg(F.percentile_approx("engine_cc", 0.5, 1000)).collect()[0][0])
vehicles = (vehicles.join(cc_group, on=["vehicle_type", "vehicle_segment"], how="left")
            .withColumn("engine_cc",
                        F.coalesce(F.col("engine_cc"), F.col("cc_med"), F.lit(cc_global)))
            .drop("cc_med"))
print("  engine_cc imputed (vehicle_type x segment median)")

if not IDV_ARTIFACT:
    idv_med = (vehicles.filter(F.col("idv").isNotNull())
               .groupBy("vehicle_make")
               .agg(F.percentile_approx("idv", 0.5, 1000).alias("idv_med")))
    vehicles = (vehicles.join(idv_med, on="vehicle_make", how="left")
                .withColumn("idv", F.coalesce(F.col("idv"), F.col("idv_med")))
                .drop("idv_med"))
    print("  idv imputed (make median)")
else:
    print(f"  idv nulls remaining: "
          f"{vehicles.filter(F.col('idv').isNull()).count():,} (expect 0)")

# ══════════════════════════════════════════════════════════════
# D. policies — reconstruct premium
# ══════════════════════════════════════════════════════════════
print("\n── D. policies ──")

recon = F.greatest(F.lit(500.0), F.round(
    F.col("base_od_premium") - F.col("ncb_discount")
    - F.col("deductible_discount") + F.col("tp_premium")
    + F.col("addon_premium"), 2))

max_diff = (policies.filter(F.col("total_premium").isNotNull())
            .select(F.max(F.abs(F.col("total_premium") - recon)).alias("d"))
            .collect()[0]["d"])
print(f"  Reconstruction test on rows that HAVE a premium:")
print(f"    max |actual - reconstructed| = ₹{max_diff:.4f}  (≈0 means exact)")

n_prem_null = policies.filter(F.col("total_premium").isNull()).count()
policies = (policies
            .withColumn("premium_was_missing",
                        F.col("total_premium").isNull().cast("int"))
            .withColumn("total_premium",
                        F.coalesce(F.col("total_premium"), recon)))
print(f"  total_premium reconstructed for {n_prem_null:,} policies")

print("\nRemaining nulls in key columns:")
policies.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["idv", "base_od_premium", "ncb_discount",
              "addon_premium", "total_premium"]
]).show()

STEP 3 — IMPOSSIBLE VALUES, EXCLUSIONS AND IMPUTATION

── A. Corrupt-label exclusion ──
  Excluded 2,022 policies (all five tables kept consistent)
  Missingness in IDV is random, so dropping ALL of them
  (claimants and non-claimants) does not bias the claim rate.

── B. policyholders ──
  Impossible ages (<18 or >85) -> null -> occupation median: 1,987
  NCB outside valid slabs                : 0
  NCB > 0 despite prior claims (rule: 0) : 0
  NCB imputed: 0 if prior claims, else clean-history median (25%)
  Income capped at P1 ₹10,599 / P99 ₹1,249,988
  Dropped PII columns: phone, email

── C. vehicles ──
  vehicle_age_years imputed (segment median) + vehicle_age_missing flag
  registration_year recomputed from cleaned age
  engine_cc imputed (vehicle_type x segment median)
  idv nulls remaining: 0 (expect 0)

── D. policies ──
  Reconstruction test on rows that HAVE a premium:
    max |actual - reconstructed| = ₹0.0000  (≈0 means exact)
  total_premium reconstructed for 1,428 polici

In [6]:
print("STEP 4 — CLAIM INTEGRITY CHECKS AND IDV RULE\n")

cp = claims.join(
    policies.select("policy_id", "policy_start_date", "policy_end_date",
                    F.col("idv").alias("pol_idv")),
    on="policy_id", how="inner"
)

# ── 1. Claim date inside policy period ────────────────────────
bad_dates = cp.filter(
    (F.col("claim_date") < F.col("policy_start_date")) |
    (F.col("claim_date") > F.col("policy_end_date"))
).count()
print(f"1. Claims dated outside their policy period : {bad_dates:,}  (must be 0)")

# ── 2. claims table vs claim_labels reconciliation ────────────
rec = labels.filter(F.col("has_claim") == 1).join(
    claims.select("policy_id", F.col("claim_amount").alias("amt_in_claims")),
    on="policy_id", how="left"
)
mismatch = rec.filter(
    F.col("amt_in_claims").isNull() |
    (F.abs(F.col("claim_amount") - F.col("amt_in_claims")) > 0.01)
).count()
zero_but_amount = labels.filter(
    (F.col("has_claim") == 0) & (F.col("claim_amount") != 0)
).count()
flag_but_zero = labels.filter(
    (F.col("has_claim") == 1) & (F.col("claim_amount") <= 0)
).count()
print(f"2. claims vs labels amount mismatches        : {mismatch:,}  (must be 0)")
print(f"   has_claim=0 but amount != 0               : {zero_but_amount:,}  (must be 0)")
print(f"   has_claim=1 but amount <= 0               : {flag_but_zero:,}  (must be 0)")

# ── 3. Payout above IDV ───────────────────────────────────────
print("\n3. Own-damage-type payouts above IDV "
      "(third_party is exempt: liability is not capped by IDV)")
over = cp.filter(
    (F.col("claim_type") != "third_party") &
    (F.col("claim_amount") > F.col("pol_idv"))
)
n_over   = over.count()
n_nontp  = cp.filter(F.col("claim_type") != "third_party").count()
excess   = over.agg(F.sum(F.col("claim_amount") - F.col("pol_idv"))).collect()[0][0] or 0
print(f"   Claims above IDV : {n_over:,} of {n_nontp:,} "
      f"({n_over/max(n_nontp,1)*100:.2f}%)")
print(f"   Total excess     : ₹{excess:,.0f}")
over.groupBy("claim_type").agg(
    F.count("*").alias("claims_over_idv"),
    F.round(F.avg(F.col("claim_amount") / F.col("pol_idv")), 3).alias("avg_claim_to_idv"),
).show()

# Apply the contractual cap to claims, then propagate to labels
claims = (
    claims.join(policies.select("policy_id", F.col("idv").alias("pol_idv")),
                on="policy_id", how="left")
    .withColumn("was_capped_at_idv",
                ((F.col("claim_type") != "third_party") &
                 (F.col("claim_amount") > F.col("pol_idv"))).cast("int"))
    .withColumn("claim_amount",
                F.when(F.col("was_capped_at_idv") == 1, F.col("pol_idv"))
                 .otherwise(F.col("claim_amount")))
    .withColumn("surveyor_assessment",
                F.when((F.col("claim_type") != "third_party") &
                       (F.col("surveyor_assessment") > F.col("pol_idv")),
                       F.col("pol_idv"))
                 .otherwise(F.col("surveyor_assessment")))
    .drop("pol_idv")
)

labels = (
    labels.join(claims.select("policy_id", F.col("claim_amount").alias("amt_new")),
                on="policy_id", how="left")
    .withColumn("claim_amount", F.coalesce(F.col("amt_new"), F.col("claim_amount")))
    .drop("amt_new")
)

claims.cache(); labels.cache(); policies.cache()

# ── 4. IDV consistency between vehicles and policies ──────────
idv_mismatch = policies.select("policy_id", "policyholder_id",
                               F.col("idv").alias("p_idv")).join(
    vehicles.select("policyholder_id", F.col("idv").alias("v_idv")),
    on="policyholder_id").filter(F.col("p_idv") != F.col("v_idv")).count()
print(f"4. vehicles.idv != policies.idv : {idv_mismatch:,}")

# ── 5. Tail check (we do NOT cap percentiles) ─────────────────
print("\n5. Claim amount tail after the IDV rule (non-zero claims):")
q = labels.filter(F.col("claim_amount") > 0).approxQuantile(
    "claim_amount", [0.5, 0.9, 0.99, 0.999, 1.0], 0.001)
for name, val in zip(["P50", "P90", "P99", "P99.9", "Max"], q):
    print(f"   {name:<6}: ₹{val:>14,.0f}")
print("   The tail is kept on purpose. These are the claims a")
print("   Tweedie model exists to price.")

tot_claims = labels.agg(F.sum("claim_amount")).collect()[0][0]
tot_prem   = policies.agg(F.sum("total_premium")).collect()[0][0]
print(f"\nLoss ratio after cleaning: {tot_claims/tot_prem*100:.2f}%  "
      f"(was 440.73% in Notebook 01)")

STEP 4 — CLAIM INTEGRITY CHECKS AND IDV RULE

1. Claims dated outside their policy period : 0  (must be 0)
2. claims vs labels amount mismatches        : 0  (must be 0)
   has_claim=0 but amount != 0               : 0  (must be 0)
   has_claim=1 but amount <= 0               : 0  (must be 0)

3. Own-damage-type payouts above IDV (third_party is exempt: liability is not capped by IDV)
   Claims above IDV : 1,273 of 26,749 (4.76%)
   Total excess     : ₹521,465,351
+----------------+---------------+----------------+
|      claim_type|claims_over_idv|avg_claim_to_idv|
+----------------+---------------+----------------+
|           theft|            592|            1.24|
|      own_damage|            314|           1.247|
|natural_calamity|            170|           1.252|
|            fire|            197|            1.24|
+----------------+---------------+----------------+

4. vehicles.idv != policies.idv : 0

5. Claim amount tail after the IDV rule (non-zero claims):
   P50   : ₹       

In [7]:
print("STEP 5 — SAVE CLEANED DATA\n")

# ── Final stability checks ────────────────────────────────────
final_rate  = labels.agg(F.avg("has_claim")).collect()[0][0] * 100
final_total = labels.agg(F.sum("claim_amount")).collect()[0][0]

print(f"Claim rate   : {INITIAL_CLAIM_RATE:.4f}% -> {final_rate:.4f}%")
print(f"Drift        : {final_rate - INITIAL_CLAIM_RATE:+.4f} pp")
print("✅ stable" if abs(final_rate - INITIAL_CLAIM_RATE) < 0.5
      else "⚠️ claim rate shifted — investigate before continuing")
print(f"Total claims : ₹{INITIAL_TOTAL_CLAIMS:,.0f} -> ₹{final_total:,.0f} "
      f"(changes expected: exclusion + IDV cap)")

# ── Referential integrity ─────────────────────────────────────
n_pol, n_lab = policies.count(), labels.count()
orphans = policies.join(labels, on="policy_id", how="left_anti").count()
print(f"\npolicies={n_pol:,}  labels={n_lab:,}  "
      f"policies without a label={orphans:,} (must be 0)")

# ── Save (tables are small: toPandas is safe) ─────────────────
# PyArrow / pandas FutureWarnings below are harmless — same as before
policyholders.toPandas().to_csv(f"{PROCESSED}/policyholders_clean.csv", index=False)
print("  ✅ policyholders_clean.csv")
vehicles.toPandas().to_csv(f"{PROCESSED}/vehicles_clean.csv", index=False)
print("  ✅ vehicles_clean.csv")
policies.toPandas().to_csv(f"{PROCESSED}/policies_clean.csv", index=False)
print("  ✅ policies_clean.csv")
claims.toPandas().to_csv(f"{PROCESSED}/claims_clean.csv", index=False)
print("  ✅ claims_clean.csv")

# The generator's TRUE claim probability is split off: never a feature.
labels.drop("claim_prob").toPandas().to_csv(
    f"{PROCESSED}/labels_clean.csv", index=False)
labels.select("policy_id", "claim_prob").toPandas().to_csv(
    f"{PROCESSED}/oracle_claim_prob.csv", index=False)
print("  ✅ labels_clean.csv        (target only)")
print("  ✅ oracle_claim_prob.csv   (evaluation-only, NEVER a feature)")

print("\nProcessed folder:")
for f in sorted(os.listdir(PROCESSED)):
    fp = f"{PROCESSED}/{f}"
    if os.path.isfile(fp):
        print(f"  {f:<32} {os.path.getsize(fp)/1024/1024:>8.1f} MB")

print("""
CLEANING LOG
  Duplicates         : 500 policyholder rows removed
  Types              : dates cast, 0/1 flags validated, exposure verified
  Label corruption   : null-IDV policies excluded (if confirmed in Cell 3)
  IDV payout rule    : own-damage-type claims capped at IDV, flagged
  Premium            : reconstructed from components, not imputed
  NCB                : business-rule imputation + ncb_missing flag
  Vehicle age        : segment median + vehicle_age_missing flag
  PII                : phone, email dropped
  Tail               : deliberately NOT capped
  Oracle probability : separated from the label file
""")

STEP 5 — SAVE CLEANED DATA

Claim rate   : 36.4780% -> 36.4868%
Drift        : +0.0088 pp
✅ stable
Total claims : ₹8,339,565,569 -> ₹7,817,722,364 (changes expected: exclusion + IDV cap)

policies=97,978  labels=97,978  policies without a label=0 (must be 0)
  ✅ policyholders_clean.csv
  ✅ vehicles_clean.csv
  ✅ policies_clean.csv
  ✅ claims_clean.csv
  ✅ labels_clean.csv        (target only)
  ✅ oracle_claim_prob.csv   (evaluation-only, NEVER a feature)

Processed folder:
  01_claim_distribution.png             0.2 MB
  claims_clean.csv                      3.1 MB
  labels_clean.csv                      1.9 MB
  oracle_claim_prob.csv                 2.0 MB
  policies_clean.csv                   14.2 MB
  policyholders_clean.csv               6.2 MB
  vehicles_clean.csv                    7.9 MB

CLEANING LOG
  Duplicates         : 500 policyholder rows removed
  Types              : dates cast, 0/1 flags validated, exposure verified
  Label corruption   : null-IDV policies excluded (i